# Séance 6 – Classes et instances (1)
### Lab : premières classes – **CORRIGÉ (version formateur)**

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 4 : la classe Equipment |

**Objectifs**
- Traduire un diagramme de classes UML en code
- Écrire une classe avec __init__, des attributs et des méthodes
- Instancier des objets et les faire collaborer

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

---
## Partie 1 – Classes, constructeur et attributs

> **Rappel**
> ```python
> class NomDeClasse:
>     def __init__(self, param: type) -> None:
>         self.attribut = param          # attribut d'instance
>
>     def methode(self, autre: type) -> type:
>         return ...                     # utilise self.attribut
>
> objet = NomDeClasse(valeur)            # instanciation
> objet.attribut, objet.methode(x)       # accès
> ```
> - Tous les attributs sont créés dans `__init__` · `vars(objet)` affiche les attributs

### Exemple guidé : un serveur supervisé

In [1]:
class Server:
    """Serveur supervisé."""

    def __init__(self, hostname: str, cpu_count: int, ram_gb: int) -> None:
        self.hostname = hostname
        self.cpu_count = cpu_count
        self.ram_gb = ram_gb
        self.services: list[str] = []

    def add_service(self, name: str) -> None:
        """Déclare un service hébergé sur le serveur."""
        if name not in self.services:
            self.services.append(name)

    def ram_per_cpu(self) -> float:
        """Mémoire disponible par cœur, en Go."""
        return self.ram_gb / self.cpu_count


web = Server("srv-web-01", 8, 32)
db = Server("srv-db-01", 16, 128)
web.add_service("nginx")
web.add_service("nginx")
db.add_service("postgresql")
print(web.hostname, web.services, web.ram_per_cpu())
print(db.hostname, db.services, db.ram_per_cpu())
print(vars(web))

srv-web-01 ['nginx'] 4.0
srv-db-01 ['postgresql'] 8.0
{'hostname': 'srv-web-01', 'cpu_count': 8, 'ram_gb': 32, 'services': ['nginx']}


### Exercice 1 ⭐ – La classe Employee

Écrivez la classe `Employee` :
- attributs `first_name`, `last_name`, `department` ;
- méthode `full_name()` → `"Amira BEN SALAH"` (nom de famille en majuscules) ;
- méthode `email(domain: str = "entreprise.tn")` → `"amira.ben-salah@entreprise.tn"` (minuscules, espaces du nom remplacés par `-`).

In [2]:
class Employee:
    """Employé de l'entreprise."""

    def __init__(self, first_name: str, last_name: str, department: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.department = department

    def full_name(self) -> str:
        """Prénom et nom (nom en majuscules)."""
        return f"{self.first_name} {self.last_name.upper()}"

    def email(self, domain: str = "entreprise.tn") -> str:
        """Adresse e-mail professionnelle."""
        local = f"{self.first_name}.{self.last_name}".lower().replace(" ", "-")
        return f"{local}@{domain}"


amira = Employee("Amira", "Ben Salah", "DSI")
print(amira.full_name(), amira.email())

Amira BEN SALAH amira.ben-salah@entreprise.tn


<details><summary>💡 Indice</summary>

N'oubliez pas `self` en premier paramètre de chaque méthode, et `self.` devant chaque attribut.

</details>

In [3]:
# ✅ Vérification
amira = Employee("Amira", "Ben Salah", "DSI")
assert amira.department == "DSI"
assert amira.full_name() == "Amira BEN SALAH"
assert amira.email() == "amira.ben-salah@entreprise.tn"
assert Employee("Karim", "Trabelsi", "RH").email("groupe.com") == "karim.trabelsi@groupe.com"
print("✅ Exercice 1 réussi")

✅ Exercice 1 réussi


### Exercice 2 ⭐ – Manipuler plusieurs objets

1. Créez la liste `staff` de trois employés : Amira Ben Salah (DSI), Karim Trabelsi (Finance), Sonia Haddad (DSI).
2. Construisez `dsi_emails` : la liste des e-mails des employés de la DSI (compréhension).
3. Karim change de service : passez son département à `"DSI"`, puis calculez `dsi_count`.

In [4]:
staff = [
    Employee("Amira", "Ben Salah", "DSI"),
    Employee("Karim", "Trabelsi", "Finance"),
    Employee("Sonia", "Haddad", "DSI"),
]
dsi_emails = [e.email() for e in staff if e.department == "DSI"]
staff[1].department = "DSI"
dsi_count = len([e for e in staff if e.department == "DSI"])
print(dsi_emails, dsi_count)

['amira.ben-salah@entreprise.tn', 'sonia.haddad@entreprise.tn'] 3


<details><summary>💡 Indice</summary>

Une liste peut contenir des objets : `[Employee(...), Employee(...)]`. On accède à leurs attributs avec `e.department`.

</details>

In [5]:
# ✅ Vérification
assert len(staff) == 3 and all(isinstance(e, Employee) for e in staff)
assert dsi_emails == ["amira.ben-salah@entreprise.tn", "sonia.haddad@entreprise.tn"]
assert staff[1].department == "DSI" and dsi_count == 3
print("✅ Exercice 2 réussi")

✅ Exercice 2 réussi


### Exercice 3 ⭐⭐ – Un volume disque et ses règles

Écrivez la classe `DiskVolume(mount_point: str, total_gb: float, used_gb: float = 0.0)` :
- `free_gb()` : espace libre ;
- `usage_percent()` : taux d'occupation arrondi à 1 décimale ;
- `store(size_gb)` → `bool` : ajoute les données si l'espace libre suffit, sinon ne change rien et renvoie `False` ;
- `cleanup(size_gb)` → `float` : libère au plus `size_gb` (sans passer sous 0) et renvoie l'espace réellement libéré.

In [6]:
class DiskVolume:
    """Volume disque monté sur un serveur."""

    def __init__(self, mount_point: str, total_gb: float, used_gb: float = 0.0) -> None:
        self.mount_point = mount_point
        self.total_gb = total_gb
        self.used_gb = used_gb

    def free_gb(self) -> float:
        """Espace libre en Go."""
        return self.total_gb - self.used_gb

    def usage_percent(self) -> float:
        """Taux d'occupation en %."""
        return round(self.used_gb / self.total_gb * 100, 1)

    def store(self, size_gb: float) -> bool:
        """Stocke des données si l'espace le permet."""
        if size_gb > self.free_gb():
            return False
        self.used_gb += size_gb
        return True

    def cleanup(self, size_gb: float) -> float:
        """Libère au plus size_gb et renvoie l'espace réellement libéré."""
        freed = min(size_gb, self.used_gb)
        self.used_gb -= freed
        return freed


var = DiskVolume("/var", 100, 62.5)
print(var.free_gb(), var.usage_percent(), var.store(40), var.store(30), var.usage_percent())
print(var.cleanup(200), var.used_gb)

37.5 62.5 False True 92.5
92.5 0.0


<details><summary>💡 Indice</summary>

Une méthode peut appeler une autre méthode du même objet : `self.free_gb()`.

</details>

In [7]:
# ✅ Vérification
vol = DiskVolume("/data", 500)
assert vol.used_gb == 0.0 and vol.free_gb() == 500
assert vol.store(450) is True and vol.usage_percent() == 90.0
assert vol.store(60) is False and vol.used_gb == 450
assert vol.cleanup(100) == 100 and vol.used_gb == 350
assert vol.cleanup(1000) == 350 and vol.used_gb == 0
print("✅ Exercice 3 réussi")

✅ Exercice 3 réussi


### Exercice 4 ⭐⭐ – Du diagramme UML au code

Implémentez la classe `Ticket` décrite par ce diagramme :

![UML Ticket](img/s06_ticket.svg)

- `assign(technician)` : enregistre le technicien et passe le statut à `"ASSIGNED"` ;
- `add_comment(text)` : ajoute un commentaire ;
- `close(resolution)` : impossible (renvoie `False`) si aucun technicien n'est affecté ; sinon ajoute le commentaire `"Résolution : ..."`, passe le statut à `"CLOSED"` et renvoie `True` ;
- `summary()` : `"[INC-0042] P2 Imprimante hors ligne – CLOSED – 2 commentaire(s)"`.

⚠️ Chaque ticket doit avoir **sa propre** liste de commentaires.

In [8]:
class Ticket:
    """Ticket d'incident."""

    def __init__(self, ticket_id: str, title: str, priority: str) -> None:
        self.ticket_id = ticket_id
        self.title = title
        self.priority = priority
        self.status = "NEW"
        self.technician: str | None = None
        self.comments: list[str] = []

    def assign(self, technician: str) -> None:
        """Affecte le ticket à un technicien."""
        self.technician = technician
        self.status = "ASSIGNED"

    def add_comment(self, text: str) -> None:
        """Ajoute un commentaire au ticket."""
        self.comments.append(text)

    def close(self, resolution: str) -> bool:
        """Clôture le ticket s'il est affecté."""
        if self.technician is None:
            return False
        self.add_comment(f"Résolution : {resolution}")
        self.status = "CLOSED"
        return True

    def summary(self) -> str:
        """Résumé d'une ligne."""
        return f"[{self.ticket_id}] {self.priority} {self.title} – {self.status} – {len(self.comments)} commentaire(s)"


t = Ticket("INC-0042", "Imprimante hors ligne", "P2")
print(t.close("redémarrage"))
t.assign("Youssef")
t.add_comment("Câble réseau débranché")
print(t.close("câble rebranché"), t.summary())

False
True [INC-0042] P2 Imprimante hors ligne – CLOSED – 2 commentaire(s)


<details><summary>💡 Indice</summary>

Les attributs avec une valeur initiale fixe (`status`, `technician`, `comments`) ne sont pas des paramètres de `__init__` : on les initialise directement.

</details>

In [9]:
# ✅ Vérification
a = Ticket("INC-1", "Écran noir", "P3")
b = Ticket("INC-2", "VPN lent", "P2")
a.add_comment("Vu avec l'utilisateur")
assert b.comments == [], "les commentaires sont partagés entre tickets !"
assert a.status == "NEW" and a.close("x") is False
a.assign("Sonia")
assert a.status == "ASSIGNED" and a.technician == "Sonia"
assert a.close("écran remplacé") is True and a.status == "CLOSED"
assert a.comments[-1] == "Résolution : écran remplacé"
assert a.summary() == "[INC-1] P3 Écran noir – CLOSED – 2 commentaire(s)"
print("✅ Exercice 4 réussi")

✅ Exercice 4 réussi


---
## Partie 2 – Méthodes et objets qui collaborent

> **Rappel**
> - Une méthode = une fonction de la classe, premier paramètre `self`
> - Une méthode peut appeler une autre méthode : `self.autre()`
> - Un attribut peut contenir d'autres objets (liste d'objets, dictionnaire d'objets)

### Exemple guidé : un employé reçoit des équipements

In [10]:
class Laptop:
    def __init__(self, asset_tag: str) -> None:
        self.asset_tag = asset_tag
        self.owner: str | None = None


class Staff:
    def __init__(self, name: str) -> None:
        self.name = name
        self.laptops: list[Laptop] = []

    def receive(self, laptop: Laptop) -> bool:
        if laptop.owner is not None:
            return False
        laptop.owner = self.name
        self.laptops.append(laptop)
        return True


lt = Laptop("LT-001")
amira, karim = Staff("Amira"), Staff("Karim")
print(amira.receive(lt), karim.receive(lt), lt.owner, [l.asset_tag for l in amira.laptops])

True False Amira ['LT-001']


### Exercice 5 ⭐⭐ – Budget d'un projet

Écrivez la classe `Project(name: str, budget: float)` qui gère une liste de tâches (chaque tâche est un dictionnaire `{"title", "hours", "rate"}`) :
- `add_task(title, hours, rate=60.0)` ;
- `total_hours()` et `total_cost()` (heures × taux horaire) ;
- `remaining_budget()` ;
- `is_over_budget()` ;
- `most_expensive_task()` → titre de la tâche la plus coûteuse, ou `None` s'il n'y a aucune tâche.

In [11]:
class Project:
    """Projet avec suivi budgétaire."""

    def __init__(self, name: str, budget: float) -> None:
        self.name = name
        self.budget = budget
        self.tasks: list[dict] = []

    def add_task(self, title: str, hours: float, rate: float = 60.0) -> None:
        """Ajoute une tâche chiffrée."""
        self.tasks.append({"title": title, "hours": hours, "rate": rate})

    def total_hours(self) -> float:
        return sum([t["hours"] for t in self.tasks])

    def total_cost(self) -> float:
        return sum([t["hours"] * t["rate"] for t in self.tasks])

    def remaining_budget(self) -> float:
        return self.budget - self.total_cost()

    def is_over_budget(self) -> bool:
        return self.total_cost() > self.budget

    def most_expensive_task(self) -> str | None:
        if not self.tasks:
            return None
        return max(self.tasks, key=lambda t: t["hours"] * t["rate"])["title"]


vpn = Project("Déploiement VPN", 5000)
vpn.add_task("Étude", 16)
vpn.add_task("Installation", 24, rate=80)
vpn.add_task("Formation utilisateurs", 8, rate=50)
print(vpn.total_hours(), vpn.total_cost(), vpn.remaining_budget(), vpn.is_over_budget(), vpn.most_expensive_task())

48 3280.0 1720.0 False Installation


<details><summary>💡 Indice</summary>

Réutilisez `total_cost()` dans `remaining_budget()` et `is_over_budget()`. `max(liste, key=...)` renvoie l'élément maximal.

</details>

In [12]:
# ✅ Vérification
p = Project("Test", 1000)
assert p.most_expensive_task() is None and p.total_cost() == 0
p.add_task("A", 10)
p.add_task("B", 5, rate=100)
assert p.total_hours() == 15 and p.total_cost() == 1100
assert p.remaining_budget() == -100 and p.is_over_budget() is True
assert p.most_expensive_task() == "A"
assert Project("Autre", 10).tasks == []
print("✅ Exercice 5 réussi")

✅ Exercice 5 réussi


### Exercice 6 ⭐⭐ – Un switch réseau et ses ports

Écrivez la classe `NetworkSwitch(hostname: str, port_count: int)`. Les ports sont numérotés de 1 à `port_count` ;
l'attribut `ports` est un dictionnaire numéro → nom de l'appareil connecté (ou `None`).
- `connect(port, device)` → `bool` : `False` si le port n'existe pas ou est déjà occupé ;
- `disconnect(port)` → nom de l'appareil déconnecté, ou `None` ;
- `free_ports()` → liste des ports libres ;
- `find_device(device)` → numéro du port, ou `None` ;
- `usage()` → taux d'occupation en % (entier arrondi).

In [13]:
class NetworkSwitch:
    """Switch réseau avec gestion des ports."""

    def __init__(self, hostname: str, port_count: int) -> None:
        self.hostname = hostname
        self.port_count = port_count
        self.ports: dict[int, str | None] = {p: None for p in range(1, port_count + 1)}

    def connect(self, port: int, device: str) -> bool:
        if port not in self.ports or self.ports[port] is not None:
            return False
        self.ports[port] = device
        return True

    def disconnect(self, port: int) -> str | None:
        device = self.ports.get(port)
        if device is not None:
            self.ports[port] = None
        return device

    def free_ports(self) -> list[int]:
        return [p for p, device in self.ports.items() if device is None]

    def find_device(self, device: str) -> int | None:
        for port, connected in self.ports.items():
            if connected == device:
                return port
        return None

    def usage(self) -> int:
        used = self.port_count - len(self.free_ports())
        return round(used / self.port_count * 100)


sw = NetworkSwitch("sw-etage2", 8)
print(sw.connect(1, "PC-Amira"), sw.connect(1, "PC-Karim"), sw.connect(9, "AP-01"))
sw.connect(2, "PR-001")
print(sw.free_ports(), sw.find_device("PR-001"), sw.usage(), sw.disconnect(1), sw.usage())

True False False
[3, 4, 5, 6, 7, 8] 2 25 PC-Amira 12


<details><summary>💡 Indice</summary>

Initialisez `ports` avec une compréhension de dictionnaire. `self.ports.get(port)` renvoie `None` pour un port inexistant.

</details>

In [14]:
# ✅ Vérification
sw = NetworkSwitch("sw-test", 4)
assert sw.ports == {1: None, 2: None, 3: None, 4: None}
assert sw.connect(1, "PC-1") and sw.connect(3, "AP-1")
assert not sw.connect(3, "PC-2") and not sw.connect(5, "PC-2") and not sw.connect(0, "PC-2")
assert sw.free_ports() == [2, 4] and sw.usage() == 50
assert sw.find_device("AP-1") == 3 and sw.find_device("XX") is None
assert sw.disconnect(3) == "AP-1" and sw.disconnect(3) is None and sw.usage() == 25
print("✅ Exercice 6 réussi")

✅ Exercice 6 réussi


### Exercice 7 ⭐⭐⭐ – Journal d'une sonde de température

Une sonde de salle serveur enregistre des mesures. Écrivez la classe `TemperatureSensor(room: str, max_temp: float = 27.0)` :
- `record(value)` : enregistre une mesure ; les valeurs hors de [-20, 80] sont des erreurs de capteur et sont **ignorées** (comptées dans `errors`) ;
- `average()` (arrondie à 1 décimale, `None` si aucune mesure), `min_max()` → tuple `(min, max)` ou `None` ;
- `alerts()` : liste des positions (indices dans `readings`) où la température dépasse `max_temp` ;
- `longest_alert()` : longueur de la plus longue série **consécutive** de mesures au-dessus de `max_temp`.

In [15]:
class TemperatureSensor:
    """Sonde de température d'une salle technique."""

    def __init__(self, room: str, max_temp: float = 27.0) -> None:
        self.room = room
        self.max_temp = max_temp
        self.readings: list[float] = []
        self.errors = 0

    def record(self, value: float) -> None:
        if -20 <= value <= 80:
            self.readings.append(value)
        else:
            self.errors += 1

    def average(self) -> float | None:
        if not self.readings:
            return None
        return round(sum(self.readings) / len(self.readings), 1)

    def min_max(self) -> tuple[float, float] | None:
        if not self.readings:
            return None
        return min(self.readings), max(self.readings)

    def alerts(self) -> list[int]:
        return [i for i, value in enumerate(self.readings) if value > self.max_temp]

    def longest_alert(self) -> int:
        longest = current = 0
        for value in self.readings:
            current = current + 1 if value > self.max_temp else 0
            longest = max(longest, current)
        return longest


sensor = TemperatureSensor("Salle serveurs A")
for v in [22.5, 23.0, 27.5, 28.1, 99.9, 26.0, 27.2, 27.9, 28.4, 24.0, -40]:
    sensor.record(v)
print(sensor.average(), sensor.min_max(), sensor.alerts(), sensor.longest_alert(), sensor.errors)

26.1 (22.5, 28.4) [2, 3, 5, 6, 7] 3 2


<details><summary>💡 Indice</summary>

Pour la plus longue série : un compteur `current` remis à 0 dès qu'une mesure est normale, et un maximum `longest`.

</details>

In [16]:
# ✅ Vérification
s = TemperatureSensor("Test")
assert s.average() is None and s.min_max() is None and s.longest_alert() == 0
for v in [22.5, 23.0, 27.5, 28.1, 99.9, 26.0, 27.2, 27.9, 28.4, 24.0, -40]:
    s.record(v)
assert s.errors == 2 and len(s.readings) == 9
assert s.average() == 26.1
assert s.min_max() == (22.5, 28.4)
assert s.alerts() == [2, 3, 5, 6, 7]
assert s.longest_alert() == 3
print("✅ Exercice 7 réussi")

✅ Exercice 7 réussi


---
### Exercice de synthèse ⭐⭐⭐ – Tableau Kanban

Écrivez la classe `KanbanBoard(project: str, wip_limit: int = 3)` avec trois colonnes `"TODO"`, `"DOING"`, `"DONE"`
(attribut `columns` : dictionnaire colonne → liste de titres de tâches).
- `add(task)` : ajoute dans `TODO` ; refuse (`False`) une tâche déjà présente dans **n'importe quelle** colonne ;
- `column_of(task)` → nom de la colonne, ou `None` ;
- `move(task, to)` → `bool` : refuse une tâche inconnue, une colonne inconnue, et tout déplacement vers `DOING` si cette colonne contient déjà `wip_limit` tâches (limite de travail en cours) ;
- `progress()` → pourcentage de tâches terminées (entier arrondi, 0 si aucune tâche) ;
- `report()` → `"TODO: 2 | DOING: 1 | DONE: 3"`.

In [17]:
class KanbanBoard:
    """Tableau Kanban avec limite de travail en cours."""

    COLUMNS = ("TODO", "DOING", "DONE")

    def __init__(self, project: str, wip_limit: int = 3) -> None:
        self.project = project
        self.wip_limit = wip_limit
        self.columns: dict[str, list[str]] = {name: [] for name in self.COLUMNS}

    def column_of(self, task: str) -> str | None:
        for name, tasks in self.columns.items():
            if task in tasks:
                return name
        return None

    def add(self, task: str) -> bool:
        if self.column_of(task) is not None:
            return False
        self.columns["TODO"].append(task)
        return True

    def move(self, task: str, to: str) -> bool:
        current = self.column_of(task)
        if current is None or to not in self.columns:
            return False
        if to == "DOING" and current != "DOING" and len(self.columns["DOING"]) >= self.wip_limit:
            return False
        self.columns[current].remove(task)
        self.columns[to].append(task)
        return True

    def progress(self) -> int:
        total = sum([len(tasks) for tasks in self.columns.values()])
        return round(len(self.columns["DONE"]) / total * 100) if total else 0

    def report(self) -> str:
        return " | ".join([f"{name}: {len(tasks)}" for name, tasks in self.columns.items()])


board = KanbanBoard("Migration AD", wip_limit=2)
for task in ["Inventaire", "Plan de migration", "Tests", "Bascule", "Inventaire"]:
    board.add(task)
board.move("Inventaire", "DOING")
board.move("Plan de migration", "DOING")
print(board.move("Tests", "DOING"))
board.move("Inventaire", "DONE")
print(board.report(), board.progress())

False
TODO: 2 | DOING: 1 | DONE: 1 25


<details><summary>💡 Indice</summary>

Écrivez d'abord `column_of`, puis réutilisez-la dans `add` et `move`. `COLUMNS` est une constante partagée par la classe : les attributs de classe seront détaillés en S7.

</details>

In [18]:
# ✅ Vérification
b = KanbanBoard("Test", wip_limit=1)
assert b.progress() == 0
assert b.add("A") and b.add("B") and not b.add("A")
assert b.move("A", "DOING") and not b.move("B", "DOING")
assert not b.move("Z", "DONE") and not b.move("A", "ARCHIVE")
assert b.move("A", "DONE") and b.move("B", "DOING")
assert b.column_of("B") == "DOING" and b.column_of("Z") is None
assert b.report() == "TODO: 0 | DOING: 1 | DONE: 1" and b.progress() == 50
assert KanbanBoard("X").columns == {"TODO": [], "DOING": [], "DONE": []}
print("✅ Exercice de synthèse réussi")

✅ Exercice de synthèse réussi


---
## Projet fil rouge – étape 4 : la classe `Equipment`

Les dictionnaires et les fonctions de l'étape 3 deviennent une classe, conforme au diagramme UML :

![UML Equipment](img/s06_equipment.svg)

**À faire :**
1. Écrivez la classe `Equipment` : constructeur (numéro en majuscules, prix converti en `float`), `age(current_year)`, `is_available()`,
   `assign(employee)` (`False` si déjà affecté), `unassign()`, `describe()` (même format que `format_equipment` de S5).
2. Adaptez les fonctions de l'inventaire pour qu'elles manipulent une `list[Equipment]` : `find_equipment`, `add_equipment`, `inventory_stats`, `total_value`, `equipment_older_than(inventory, years, current_year)`.

> En S7, ces fonctions rejoindront une classe `Inventory`.

In [19]:
EQUIPMENT_CATEGORIES = {"laptop", "server", "printer", "network"}


class Equipment:
    """Équipement du parc informatique."""

    def __init__(self, asset_tag: str, category: str, brand: str, model: str,
                 purchase_year: int, price: float, assigned_to: str | None = None) -> None:
        self.asset_tag = asset_tag.upper()
        self.category = category
        self.brand = brand
        self.model = model
        self.purchase_year = purchase_year
        self.price = float(price)
        self.assigned_to = assigned_to

    def age(self, current_year: int) -> int:
        """Âge de l'équipement en années."""
        return current_year - self.purchase_year

    def is_available(self) -> bool:
        """Vrai si l'équipement n'est affecté à personne."""
        return self.assigned_to is None

    def assign(self, employee: str) -> bool:
        """Affecte l'équipement ; False s'il est déjà affecté."""
        if not self.is_available():
            return False
        self.assigned_to = employee
        return True

    def unassign(self) -> None:
        """Libère l'équipement."""
        self.assigned_to = None

    def describe(self) -> str:
        """Ligne d'affichage de l'équipement."""
        name = f"{self.brand} {self.model}"
        return (f"{self.asset_tag:<7} | {self.category:<8} | {name:<25} | "
                f"{self.price:>10,.2f} DT | {self.assigned_to or '-'}")


def find_equipment(inventory: list[Equipment], asset_tag: str) -> Equipment | None:
    """Recherche un équipement par numéro (insensible à la casse)."""
    tag = asset_tag.upper()
    for equipment in inventory:
        if equipment.asset_tag == tag:
            return equipment
    return None


def add_equipment(inventory: list[Equipment], equipment: Equipment) -> bool:
    """Ajoute un équipement si sa catégorie est connue et son numéro libre."""
    if equipment.category not in EQUIPMENT_CATEGORIES or find_equipment(inventory, equipment.asset_tag):
        return False
    inventory.append(equipment)
    return True


def inventory_stats(inventory: list[Equipment]) -> dict[str, int]:
    """Nombre d'équipements par catégorie."""
    stats: dict[str, int] = {}
    for equipment in inventory:
        stats[equipment.category] = stats.get(equipment.category, 0) + 1
    return stats


def total_value(inventory: list[Equipment], category: str | None = None) -> float:
    """Valeur totale, éventuellement pour une catégorie."""
    return sum([e.price for e in inventory if category is None or e.category == category])


def equipment_older_than(inventory: list[Equipment], years: int, current_year: int) -> list[Equipment]:
    """Équipements dont l'âge est supérieur ou égal à years."""
    return [e for e in inventory if e.age(current_year) >= years]


inventory: list[Equipment] = []
add_equipment(inventory, Equipment("lt-001", "laptop", "Dell", "Latitude 5440", 2023, 3200, "Amira"))
add_equipment(inventory, Equipment("SV-001", "server", "HPE", "ProLiant DL380", 2019, 18500))
add_equipment(inventory, Equipment("PR-001", "printer", "HP", "LaserJet M404", 2021, 950))
add_equipment(inventory, Equipment("SW-001", "network", "Cisco", "Catalyst 9200", 2020, 4100))
inventory[2].assign("Karim")
for equipment in inventory:
    print(equipment.describe())
print(inventory_stats(inventory), total_value(inventory))
print([e.asset_tag for e in equipment_older_than(inventory, 5, 2026)])

LT-001  | laptop   | Dell Latitude 5440        |   3,200.00 DT | Amira
SV-001  | server   | HPE ProLiant DL380        |  18,500.00 DT | -
PR-001  | printer  | HP LaserJet M404          |     950.00 DT | Karim
SW-001  | network  | Cisco Catalyst 9200       |   4,100.00 DT | -
{'laptop': 1, 'server': 1, 'printer': 1, 'network': 1} 26750.0
['SV-001', 'PR-001', 'SW-001']


In [20]:
# ✅ Vérification
e = Equipment("lt-099", "laptop", "Asus", "ExpertBook B5", 2022, "2500")
assert e.asset_tag == "LT-099" and e.price == 2500.0 and e.age(2026) == 4
assert e.is_available() and e.assign("Sonia") and not e.assign("Karim") and e.assigned_to == "Sonia"
e.unassign()
assert e.is_available()
assert e.describe() == "LT-099  | laptop   | Asus ExpertBook B5        |   2,500.00 DT | -"
assert len(inventory) == 4 and find_equipment(inventory, "pr-001").assigned_to == "Karim"
assert not add_equipment(inventory, Equipment("LT-001", "laptop", "X", "Y", 2024, 1))
assert not add_equipment(inventory, Equipment("TB-001", "tablet", "X", "Y", 2024, 1))
assert inventory_stats(inventory) == {"laptop": 1, "server": 1, "printer": 1, "network": 1}
assert total_value(inventory) == 26750.0 and total_value(inventory, "server") == 18500.0
assert [x.asset_tag for x in equipment_older_than(inventory, 5, 2026)] == ["SV-001", "PR-001", "SW-001"]
print("✅ Étape 4 du projet GestParc validée")

✅ Étape 4 du projet GestParc validée


✔️ Version de référence : `scripts/gestparc/main.py` (classe + fonctions + menu).

**Prochaine étape (S7)** : attributs de classe (compteur d'équipements), `from_dict`, méthodes spéciales (`__str__`, `__eq__`, `__lt__`, `__len__`…), propriétés et une classe `Inventory`.

---
## Pour aller plus loin (bonus)

1. **`vars()` et `__dict__`** : comparez `vars(e)` et `e.__dict__`. Que contient `Equipment.__dict__` ?
2. **`getattr` / `setattr`** : écrivez une boucle qui affiche les attributs `brand`, `model`, `price` d'un équipement à partir d'une liste de noms.
3. **Diagramme** : complétez le diagramme UML avec une classe `Employee` et l'association « un employé possède 0..n équipements ». Écrivez-le en Mermaid sur <https://mermaid.live>.

In [21]:
e = Equipment("LT-777", "laptop", "Dell", "XPS 13", 2024, 4200)
for name in ["brand", "model", "price"]:
    print(name, "=", getattr(e, name))
setattr(e, "price", 3990.0)
print(vars(e))

brand = Dell
model = XPS 13
price = 4200.0
{'asset_tag': 'LT-777', 'category': 'laptop', 'brand': 'Dell', 'model': 'XPS 13', 'purchase_year': 2024, 'price': 3990.0, 'assigned_to': None}
